# Personal Knowledge Base Chat

**Project 12** — Core RAG Projects

Build a system that ingests your personal notes, markdown files, and text documents,
then lets you ask natural-language questions and get grounded answers from your own knowledge base.

**Key Skills:** Document ingestion, text chunking strategies, embedding models,
vector retrieval, answer synthesis, knowledge-base management.

## Project Overview

We all accumulate knowledge in scattered notes — meeting minutes, research summaries,
book highlights, project logs, personal journals, and learning notes. Finding specific
information across hundreds of files is painful.

This notebook builds a **Personal Knowledge Base (PKB) Chat** system that:
1. **Ingests** notes from multiple file formats (`.txt`, `.md`, plain text)
2. **Chunks** documents into semantically meaningful pieces
3. **Embeds** chunks into dense vector representations
4. **Indexes** them for fast similarity search
5. **Retrieves** the most relevant chunks for a user query
6. **Generates** grounded answers with source references

### End-to-End Pipeline

```
Notes (.txt, .md) → Ingestion → Chunking → Embedding → Vector Index
                                                            ↓
User Question → Query Embedding → Similarity Search → Top-K Chunks → Answer Synthesis
```

## Learning Goals

By the end of this notebook you will understand:

1. **Ingestion** — How to load and normalize text from multiple file formats
2. **Chunking** — Different strategies (fixed-size, sentence-based, semantic) and their trade-offs
3. **Embeddings** — How text is converted to vectors, which models to choose, and why dimensionality matters
4. **Retrieval** — How similarity search works (cosine, dot-product), and the role of top-K selection
5. **Answer generation** — How to synthesize a natural answer from retrieved chunks
6. **Evaluation** — How to measure retrieval quality and answer accuracy

## Problem Statement

**Scenario:** A researcher has accumulated ~20 personal notes on various topics —
machine learning concepts, Python tips, project retrospectives, book summaries, and
meeting notes. They want to:

- Ask "What did I write about gradient descent?" and get an instant answer
- Find all notes related to a concept across files
- Get the exact source note and section cited in every answer
- See how confident the system is in its retrieval

**Goal:** Build a complete ingestion-to-answer pipeline that handles this scenario
with high retrieval accuracy and transparent citations.

## Why This Project Matters

| Pain Point | How PKB Chat Solves It |
|-----------|----------------------|
| "I know I wrote about X somewhere…" | Semantic search finds it even with different wording |
| Notes scattered across folders/apps | Unified index across all files |
| Ctrl+F only finds exact matches | Vector search finds semantically similar content |
| Reading entire files to find one fact | Chunk-level retrieval pinpoints the exact passage |
| No way to ask follow-up questions | Chat interface over your personal knowledge |

This is the foundation for tools like Obsidian AI, Notion Q&A, and personal Copilot assistants.

## Core Concepts Explained

### 1. Ingestion
Loading raw text from different file formats. Key challenges:
- Handle different encodings (UTF-8, Latin-1)
- Strip formatting artifacts (extra whitespace, markdown syntax in some cases)
- Extract metadata (filename, creation date, headings)

### 2. Chunking
Splitting documents into pieces small enough for embedding but large enough to preserve meaning.

| Strategy | Pros | Cons |
|----------|------|------|
| **Fixed-size** (e.g., 500 chars) | Simple, predictable | May split mid-sentence |
| **Sentence-based** | Respects grammar | Chunks vary wildly in size |
| **Paragraph/section** | Respects document structure | Some sections too large |
| **Semantic** | Groups related content | Complex, slower |
| **Sliding window** | Preserves boundary context | Creates redundant chunks |

### 3. Embeddings
Converting text to fixed-dimensional vectors that capture meaning.
- Similar texts → similar vectors (high cosine similarity)
- Model choice matters: `all-MiniLM-L6-v2` (fast, 384d) vs `all-mpnet-base-v2` (better, 768d)

### 4. Retrieval
Finding the K most similar chunks to a query vector.
- **Cosine similarity**: measures angle between vectors (scale-invariant)
- **Dot product**: measures angle + magnitude
- **Euclidean distance**: measures straight-line distance

### 5. Answer Generation
Combining retrieved chunks into a coherent answer:
- **Extractive**: return chunks verbatim
- **Abstractive**: rephrase/summarize (requires LLM)
- **Hybrid**: extractive with light formatting

## Environment Setup

Core dependencies:
- **sentence-transformers** — for embedding text into vectors
- **chromadb** — for vector storage and retrieval
- **numpy**, **sklearn** — for similarity computation (TF-IDF fallback)

The notebook detects available libraries and falls back to TF-IDF if
sentence-transformers or ChromaDB are not installed.

In [1]:
import os
import re
import json
import hashlib
import textwrap
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_SENTENCE_TRANSFORMERS = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available — using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_SENTENCE_TRANSFORMERS = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available — using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


## Configuration

All tuneable parameters in one place. These control chunking granularity,
retrieval depth, and embedding model selection.

In [2]:
# ── Configuration ──
CHUNK_SIZE = 400            # target characters per chunk
CHUNK_OVERLAP = 80          # overlap between consecutive chunks
TOP_K = 5                   # number of chunks to retrieve
CONFIDENCE_THRESHOLD = 0.4  # minimum score to consider a result relevant
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Chunk size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}, Top-K: {TOP_K}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Chunk size: 400, Overlap: 80, Top-K: 5
Embedding model: all-MiniLM-L6-v2


## Create Sample Knowledge Base

We create a realistic set of personal notes spanning different topics. In a real
scenario, these would be files from your filesystem — the code handles both
in-memory text and file-based loading.

Our sample knowledge base covers:
- Machine learning fundamentals
- Python programming tips
- Project retrospectives
- Book summaries
- Meeting notes

In [3]:
# ── Sample personal notes as a realistic knowledge base ──

SAMPLE_NOTES = {
    "ml_fundamentals.md": """# Machine Learning Fundamentals

## Gradient Descent
Gradient descent is an optimization algorithm used to minimize a loss function.
It works by iteratively moving in the direction of steepest descent, which is
the negative of the gradient. The learning rate controls step size.

Key variants:
- Batch GD: uses full dataset per step (slow but stable)
- Stochastic GD (SGD): uses one sample per step (fast but noisy)
- Mini-batch GD: uses a subset (best tradeoff, typical batch size 32-256)

Learning rate schedules help convergence:
- Step decay: reduce LR by factor every N epochs
- Cosine annealing: smooth LR reduction following cosine curve
- Warmup: start with small LR, ramp up, then decay

## Overfitting and Regularization
Overfitting occurs when a model learns noise in training data instead of
the underlying pattern. Signs: low training error but high validation error.

Regularization techniques:
- L1 (Lasso): adds absolute weight penalty, encourages sparsity
- L2 (Ridge): adds squared weight penalty, shrinks weights
- Dropout: randomly zeros neurons during training (typically 0.2-0.5)
- Early stopping: halt training when validation loss stops improving
- Data augmentation: artificially expand training data

## Cross-Validation
K-fold cross-validation splits data into K parts, trains on K-1, validates on 1.
Repeat K times and average results. Common K values: 5 or 10.
Stratified K-fold preserves class distribution in each fold — essential for
imbalanced datasets. Leave-one-out CV (LOOCV) uses K=N, very expensive.

## Bias-Variance Tradeoff
- High bias = underfitting (model too simple)
- High variance = overfitting (model too complex)
- Sweet spot: enough complexity to capture patterns, not noise
Ensemble methods (bagging, boosting) help reduce variance.
""",

    "python_tips.md": """# Python Tips and Best Practices

## List Comprehensions
List comprehensions are faster and more Pythonic than for loops for simple transforms.
Use them for filtering and mapping. Avoid nested comprehensions for readability.
Example: [x**2 for x in range(10) if x % 2 == 0]

## Context Managers
Use `with` statements for file handling, database connections, and locks.
Custom context managers via __enter__/__exit__ or @contextmanager decorator.
They guarantee cleanup even if exceptions occur.

## Type Hints
Type hints improve code readability and enable static analysis with mypy.
Use typing module for complex types: List[str], Dict[str, int], Optional[str].
Python 3.10+ supports union syntax: str | None instead of Optional[str].

## Virtual Environments
Always use virtual environments for project isolation.
Options: venv (built-in), conda, poetry, pdm.
Pin dependencies with pip freeze or poetry.lock for reproducibility.
Never install packages globally — it breaks other projects.

## Debugging Tips
- Use breakpoint() for interactive debugging (Python 3.7+)
- pdb commands: n (next), s (step into), c (continue), p (print), l (list)
- Use logging module instead of print() for production code
- Set DEBUG level for development, WARNING for production

## Performance Tips
- Use generators for large datasets: (x for x in data) instead of [x for x in data]
- Profile before optimizing: cProfile, line_profiler, memory_profiler
- Use numpy/pandas vectorized operations instead of Python loops
- functools.lru_cache for memoizing expensive function calls
""",

    "project_retro_q3.md": """# Q3 2024 Project Retrospective — ML Pipeline Overhaul

## What Went Well
- Migrated from scikit-learn to PyTorch for model training — 3x faster on GPU
- Implemented automated data validation with Great Expectations
- Reduced model deployment time from 2 hours to 15 minutes with Docker
- Team adopted code review practice — caught 12 bugs before production

## What Went Wrong
- Underestimated data migration complexity — took 3 weeks instead of 1
- Feature store had schema drift issues in month 2
- CI/CD pipeline broke 4 times due to Python version conflicts
- Documentation was not updated during the rush to deadline

## Action Items
- Set up automatic schema validation for feature store
- Pin Python version in all CI/CD configs
- Schedule documentation sprints every 2 weeks
- Create a data migration checklist for future projects

## Key Metrics
- Model accuracy improved from 87.2% to 91.5%
- Inference latency reduced from 120ms to 45ms
- Data pipeline failures reduced from 8/month to 1/month
- Test coverage increased from 62% to 84%
""",

    "book_notes_designing_ml_systems.md": """# Book Notes: Designing Machine Learning Systems (Chip Huyen)

## Chapter 3: Data Engineering Fundamentals
Data is the foundation of ML systems. Key considerations:
- Data sources: user activity, logs, databases, third-party APIs
- Data formats: row-based (CSV, JSON) vs columnar (Parquet, ORC)
- Data models: relational vs document vs graph
- ETL vs ELT: transform before or after loading into warehouse

Parquet is preferred for ML workloads because:
1. Columnar storage enables reading only needed features
2. Efficient compression (snappy, gzip)
3. Schema enforcement prevents data type issues

## Chapter 5: Feature Engineering
Good features > good algorithms. Feature engineering guidelines:
- Start with domain knowledge — what would a human expert use?
- Handle missing values explicitly (impute, flag, or drop)
- Encode categorical variables appropriately (one-hot for low cardinality,
  target encoding for high cardinality)
- Create interaction features when domain suggests relationships
- Use time-based features for temporal data (day of week, hour, lag features)

Feature stores enable reuse across teams and prevent training-serving skew.

## Chapter 9: Continual Learning
Models degrade over time due to data drift and concept drift.
- Data drift: input distribution changes (new user demographics)
- Concept drift: relationship between features and target changes
- Label drift: distribution of labels changes

Monitoring strategies:
- Track feature distributions over time (PSI, KS test)
- Monitor model performance metrics with alerts
- Set up retraining triggers when drift exceeds threshold
- Shadow deployment: run new model alongside old one before switching
""",

    "meeting_notes_2024_03.md": """# Meeting Notes — March 2024

## March 5: Data Team Sync
Attendees: Alice, Bob, Charlie, Diana
Topics discussed:
- Data warehouse migration to Snowflake is 60% complete
- New data sources from mobile app need ingestion pipeline
- Alice will lead the data quality dashboard initiative
- Bob raised concern about PII in analytics tables — need masking

Action items:
- Alice: data quality dashboard prototype by March 15
- Bob: PII audit of top 20 analytics tables by March 10
- Charlie: set up Snowflake staging environment by March 8
- Diana: document current ETL jobs for migration planning

## March 12: ML Model Review
Attendees: Alice, Eve, Frank
Topics discussed:
- Recommendation model v3 shows 8% improvement in click-through rate
- A/B test results for pricing model are inconclusive — need more data
- Eve proposed using embeddings for user similarity instead of collaborative filtering
- Need to evaluate model fairness across demographic groups

Decisions:
- Ship recommendation model v3 to 50% of users for extended A/B test
- Extend pricing model A/B test for 2 more weeks
- Eve will prototype embedding-based similarity approach
- Frank will run fairness audit using AI Fairness 360

## March 19: Sprint Planning
Attendees: Full team
Sprint 7 priorities:
1. Complete Snowflake migration for sales data
2. Deploy recommendation model v3
3. Build automated retraining pipeline for churn prediction
4. Fix data pipeline failures in user event ingestion
Story points allocated: 34 (team capacity: 40)
""",

    "ml_deployment_guide.md": """# ML Model Deployment Guide

## Deployment Patterns
Three main patterns for serving ML models:

### Batch Prediction
- Run model on a schedule (hourly, daily)
- Store predictions in database for serving
- Best for: recommendation systems, daily forecasts, batch scoring
- Pros: simple, cost-effective for large volumes
- Cons: predictions can be stale, not real-time

### Online Prediction (REST API)
- Model wrapped in API, predictions on request
- Frameworks: FastAPI, Flask, TensorFlow Serving, TorchServe
- Best for: real-time decisions, personalization, fraud detection
- Pros: fresh predictions, low latency possible
- Cons: requires scaling infrastructure, cold start issues

### Edge Deployment
- Model runs on device (phone, IoT, browser)
- Frameworks: TFLite, ONNX Runtime, CoreML
- Best for: offline capability, privacy-sensitive, low-latency
- Pros: no server costs, works offline
- Cons: model size constraints, harder to update

## Model Optimization for Deployment
- Quantization: reduce precision (FP32 to INT8) — 2-4x speedup
- Pruning: remove unimportant weights — smaller model
- Knowledge distillation: train small model to mimic large one
- ONNX export: framework-agnostic representation for portability

## Monitoring in Production
Key metrics to track:
- Prediction latency (p50, p95, p99)
- Prediction volume and error rates
- Feature drift using PSI or KS statistics
- Model performance on labeled data samples
- Infrastructure: CPU/GPU utilization, memory, disk
Set up alerts for anomalies in any of these metrics.
""",

    "python_async_notes.md": """# Python Async Programming Notes

## asyncio Basics
asyncio enables concurrent I/O-bound operations without threads.
Key concepts:
- Coroutines: functions defined with `async def`
- Event loop: scheduler that runs coroutines
- await: yields control back to event loop until result is ready
- Tasks: scheduled coroutines that run concurrently

## When to Use Async
Use async for I/O-bound tasks: HTTP requests, database queries, file I/O.
Do NOT use async for CPU-bound tasks — use multiprocessing instead.

Common pattern:
- Create multiple coroutines for independent I/O operations
- Gather them with asyncio.gather() to run concurrently
- Result: N requests in time of 1 (not N)

## Common Mistakes
- Mixing sync and async code without proper bridges
- Forgetting to await a coroutine (it returns a coroutine object, not the result)
- Using time.sleep() instead of asyncio.sleep() (blocks the event loop)
- Running CPU-intensive code in async functions (blocks everything)
- Creating too many concurrent connections (respect rate limits)

## Libraries
- aiohttp: async HTTP client/server
- asyncpg: async PostgreSQL client
- motor: async MongoDB driver
- httpx: modern HTTP client with both sync and async support
""",

    "research_notes_transformers.md": """# Research Notes: Transformers Architecture

## Self-Attention Mechanism
Self-attention allows each position in a sequence to attend to all other positions.
Computation: Q, K, V matrices derived from input embeddings.
Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V

Why sqrt(d_k)? Prevents dot products from growing too large in high dimensions,
which would push softmax into regions with tiny gradients.

## Multi-Head Attention
Instead of single attention, use H parallel attention heads.
Each head learns different relationship patterns:
- Head 1 might learn syntactic dependencies
- Head 2 might learn semantic similarity
- Head 3 might learn positional relationships
Outputs are concatenated and projected back to model dimension.

## Positional Encoding
Transformers have no inherent notion of sequence order (unlike RNNs).
Positional encodings add order information:
- Sinusoidal: fixed, uses sin/cos at different frequencies
- Learned: trainable embeddings per position (BERT, GPT)
- RoPE (Rotary): rotates embeddings based on position (modern LLMs)
- ALiBi: adds linear bias to attention scores based on distance

## Key Innovations Since the Original Paper
- BERT (2018): bidirectional pre-training with masked language modeling
- GPT-2/3 (2019/2020): autoregressive generation at scale
- Vision Transformer ViT (2020): patches as tokens for images
- Mixture of Experts MoE (2022+): sparse activation for efficiency
- Flash Attention (2022): IO-aware exact attention, 2-4x faster
- KV Cache: stores key-value pairs to avoid recomputation during generation
""",

    "personal_goals_2024.md": """# Personal Goals 2024

## Technical Skills
- Complete deep learning specialization — target: June 2024
- Learn Rust basics for systems programming — target: August 2024
- Build 3 end-to-end ML projects with deployment — target: December 2024
- Contribute to 2 open-source projects — target: ongoing

## Career
- Present at one tech conference or meetup
- Write 6 technical blog posts (one every 2 months)
- Take on a mentoring role for junior team members
- Complete AWS ML Specialty certification

## Reading List
- "Designing Machine Learning Systems" by Chip Huyen — DONE
- "Machine Learning Engineering" by Andriy Burkov
- "Building Machine Learning Pipelines" by Hannes Hapke
- "Reliable Machine Learning" by Cathy Chen et al.
- "The Staff Engineer's Path" by Tanya Reilly

## Health & Wellness
- Exercise 4x per week minimum
- Daily meditation practice (10 min)
- Read 30 minutes before bed instead of screens
- Take one full week vacation each quarter
""",

    "debugging_war_stories.md": """# Debugging War Stories

## The Silent NaN
Production model started returning NaN predictions for 5% of users.
Root cause: a new feature had missing values that weren't caught by validation.
The missing values propagated through the model as NaN.
Fix: added explicit NaN checks in the feature pipeline before model inference.
Lesson: validate every feature at inference time, not just training time.

## The Timezone Bug
Model trained on UTC timestamps but served with local timestamps.
Predictions were consistently wrong by 5-8 hours offset.
Root cause: front-end sent local time, backend assumed UTC.
Fix: standardize all timestamps to UTC at ingestion, convert for display only.
Lesson: always store and process in UTC. Convert only at the presentation layer.

## The Leaky Validation
Model showed 99.5% accuracy in development but 71% in production.
Root cause: target leakage — a feature derived from the target variable was
accidentally included in training. The feature was a status code that was only
set after the event we were trying to predict.
Fix: removed the leaked feature, re-trained. Honest accuracy: 84%.
Lesson: always check feature availability at prediction time, not just correlation.

## The Memory Leak
ML service gradually consumed more memory until it crashed every 12 hours.
Root cause: model was loaded into memory on every request instead of once.
Each request appended to a list that was never cleared.
Fix: load model once at startup, reuse across requests. Clear temp buffers.
Lesson: profile memory usage under sustained load, not just single requests.
""",
}

# Write notes to temporary directory for file-based ingestion demo
NOTES_DIR = os.path.join(SAVE_DIR, "_sample_notes")
os.makedirs(NOTES_DIR, exist_ok=True)

for filename, content in SAMPLE_NOTES.items():
    filepath = os.path.join(NOTES_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip())

print(f"Created {len(SAMPLE_NOTES)} sample notes in {NOTES_DIR}/")
for fn in sorted(SAMPLE_NOTES.keys()):
    size = len(SAMPLE_NOTES[fn].strip())
    print(f"  {fn} ({size:,} chars)")

Created 10 sample notes in E:\Github\Machine-Learning-Projects\GenAI\12_Personal_Knowledge_Base_Chat\_sample_notes/
  book_notes_designing_ml_systems.md (1,682 chars)
  debugging_war_stories.md (1,586 chars)
  meeting_notes_2024_03.md (1,510 chars)
  ml_deployment_guide.md (1,540 chars)
  ml_fundamentals.md (1,782 chars)
  personal_goals_2024.md (958 chars)
  project_retro_q3.md (1,047 chars)
  python_async_notes.md (1,215 chars)
  python_tips.md (1,562 chars)
  research_notes_transformers.md (1,567 chars)


## Step 1: Ingestion Pipeline

Ingestion reads files from disk, extracts text, and attaches metadata.
Good ingestion should:
- Handle multiple file formats (`.txt`, `.md`)
- Extract metadata from filenames and content (headings, dates)
- Normalize whitespace and encoding
- Be idempotent — same file produces same result every time

In [4]:
# ── Ingestion: load files with metadata ──

class Note:
    """Represents an ingested note with content and metadata."""

    def __init__(self, filepath, text):
        self.filepath = filepath
        self.filename = os.path.basename(filepath)
        self.text = text
        self.title = self._extract_title()
        self.headings = self._extract_headings()
        self.word_count = len(text.split())
        self.char_count = len(text)
        # Deterministic ID from content
        self.note_id = hashlib.md5(f"{self.filename}:{self.char_count}".encode()).hexdigest()[:10]

    def _extract_title(self):
        """Extract title from first heading or filename."""
        for line in self.text.split("\n"):
            line = line.strip()
            if line.startswith("# ") and not line.startswith("## "):
                return line.lstrip("# ").strip()
        return self.filename.replace("_", " ").replace(".md", "").replace(".txt", "").title()

    def _extract_headings(self):
        """Extract markdown headings for section metadata."""
        headings = []
        for line in self.text.split("\n"):
            line = line.strip()
            if line.startswith("#"):
                level = len(line) - len(line.lstrip("#"))
                text = line.lstrip("# ").strip()
                if text:
                    headings.append({"level": level, "text": text})
        return headings

    @property
    def metadata(self):
        return {
            "note_id": self.note_id,
            "filename": self.filename,
            "title": self.title,
            "word_count": self.word_count,
            "heading_count": len(self.headings),
        }


def ingest_directory(directory):
    """Load all .txt and .md files from a directory."""
    notes = []
    for fname in sorted(os.listdir(directory)):
        if not fname.endswith((".txt", ".md")):
            continue
        fpath = os.path.join(directory, fname)
        with open(fpath, "r", encoding="utf-8") as f:
            text = f.read().strip()
        if text:
            notes.append(Note(fpath, text))
    return notes


# Ingest our sample notes
notes = ingest_directory(NOTES_DIR)

print(f"Ingested {len(notes)} notes\n")
total_words = 0
for note in notes:
    print(f"  [{note.note_id}] {note.title}")
    print(f"    File: {note.filename} | Words: {note.word_count} | Headings: {len(note.headings)}")
    total_words += note.word_count

print(f"\nTotal words in knowledge base: {total_words:,}")

Ingested 10 notes

  [b3dddfe60e] Book Notes: Designing Machine Learning Systems (Chip Huyen)
    File: book_notes_designing_ml_systems.md | Words: 247 | Headings: 4
  [8527242c32] Debugging War Stories
    File: debugging_war_stories.md | Words: 250 | Headings: 5
  [d61e0bfffc] Meeting Notes — March 2024
    File: meeting_notes_2024_03.md | Words: 245 | Headings: 4
  [abc74f39c4] ML Model Deployment Guide
    File: ml_deployment_guide.md | Words: 230 | Headings: 7
  [8efe1b8c0b] Machine Learning Fundamentals
    File: ml_fundamentals.md | Words: 270 | Headings: 5
  [e062a1c4b0] Personal Goals 2024
    File: personal_goals_2024.md | Words: 162 | Headings: 5
  [5a99c16a2e] Q3 2024 Project Retrospective — ML Pipeline Overhaul
    File: project_retro_q3.md | Words: 179 | Headings: 5
  [cc4a499da9] Python Async Programming Notes
    File: python_async_notes.md | Words: 187 | Headings: 5
  [a34fa50808] Python Tips and Best Practices
    File: python_tips.md | Words: 230 | Headings: 7
  [a7f

## Knowledge Base Inspection

Let's examine the structure and content distribution of our knowledge base.

In [5]:
# ── Knowledge base statistics ──

word_counts = [n.word_count for n in notes]
heading_counts = [len(n.headings) for n in notes]

print("=== Knowledge Base Statistics ===")
print(f"  Total notes: {len(notes)}")
print(f"  Total words: {sum(word_counts):,}")
print(f"  Avg words/note: {np.mean(word_counts):.0f}")
print(f"  Min/Max words: {min(word_counts)} / {max(word_counts)}")
print(f"  Total headings: {sum(heading_counts)}")

# Topic distribution (by filename prefix)
topics = defaultdict(int)
for note in notes:
    prefix = note.filename.split("_")[0]
    topics[prefix] += 1

print(f"\nTopic groups (by filename prefix):")
for topic, count in sorted(topics.items()):
    print(f"  {topic}: {count} notes")

# Heading structure sample
print(f"\nSample headings from first note:")
for h in notes[0].headings[:6]:
    indent = "  " * h["level"]
    print(f"  {indent}{'#' * h['level']} {h['text']}")

=== Knowledge Base Statistics ===
  Total notes: 10
  Total words: 2,224
  Avg words/note: 222
  Min/Max words: 162 / 270
  Total headings: 52

Topic groups (by filename prefix):
  book: 1 notes
  debugging: 1 notes
  meeting: 1 notes
  ml: 2 notes
  personal: 1 notes
  project: 1 notes
  python: 2 notes
  research: 1 notes

Sample headings from first note:
    # Book Notes: Designing Machine Learning Systems (Chip Huyen)
      ## Chapter 3: Data Engineering Fundamentals
      ## Chapter 5: Feature Engineering
      ## Chapter 9: Continual Learning


## Step 2: Chunking

We implement **three chunking strategies** and compare them:

1. **Fixed-size** — Split every N characters with overlap
2. **Section-based** — Split on markdown headings
3. **Hybrid** — Split on sections first, then apply fixed-size to large sections

Each chunk carries its parent note's metadata plus chunk-level info.

In [6]:
# ── Chunk data class ──

class Chunk:
    """A text chunk with metadata lineage back to its source note."""

    def __init__(self, text, note, section_heading, chunk_index, total_chunks, strategy):
        self.text = text
        self.note_id = note.note_id
        self.filename = note.filename
        self.title = note.title
        self.section_heading = section_heading
        self.chunk_index = chunk_index
        self.total_chunks = total_chunks
        self.strategy = strategy
        # Unique deterministic ID
        content_hash = hashlib.md5(
            f"{self.note_id}:{section_heading}:{chunk_index}:{strategy}".encode()
        ).hexdigest()[:8]
        self.chunk_id = f"{self.note_id[:6]}-{content_hash}"

    @property
    def metadata(self):
        return {
            "chunk_id": self.chunk_id,
            "note_id": self.note_id,
            "filename": self.filename,
            "title": self.title,
            "section_heading": self.section_heading,
            "chunk_index": self.chunk_index,
            "total_chunks": self.total_chunks,
            "strategy": self.strategy,
        }

    @property
    def citation(self):
        sec = f", {self.section_heading}" if self.section_heading else ""
        return f"[{self.filename}]{sec} (chunk {self.chunk_index+1}/{self.total_chunks})"

    def __repr__(self):
        return f"Chunk({self.chunk_id}, {len(self.text)} chars)"


In [7]:
# ── Chunking strategies ──

def chunk_fixed_size(note, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Strategy 1: Fixed-size sliding window chunks."""
    text = note.text
    chunks = []
    start = 0
    parts = []
    while start < len(text):
        end = start + chunk_size
        parts.append(text[start:end].strip())
        start += chunk_size - overlap

    for i, part in enumerate(parts):
        if part:
            chunks.append(Chunk(part, note, "", i, len(parts), "fixed"))
    return chunks


def chunk_by_section(note, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Strategy 2: Section-based chunking using markdown headings."""
    lines = note.text.split("\n")
    sections = []
    current_heading = note.title
    current_lines = []

    for line in lines:
        if line.strip().startswith("## "):
            if current_lines:
                sections.append((current_heading, "\n".join(current_lines).strip()))
            current_heading = line.strip().lstrip("# ").strip()
            current_lines = []
        else:
            current_lines.append(line)

    if current_lines:
        sections.append((current_heading, "\n".join(current_lines).strip()))

    # For sections longer than chunk_size, sub-chunk them
    chunks = []
    for heading, section_text in sections:
        if not section_text:
            continue
        if len(section_text) <= chunk_size:
            chunks.append(Chunk(section_text, note, heading, 0, 1, "section"))
        else:
            start = 0
            parts = []
            while start < len(section_text):
                end = start + chunk_size
                parts.append(section_text[start:end].strip())
                start += chunk_size - overlap
            for i, part in enumerate(parts):
                if part:
                    chunks.append(Chunk(part, note, heading, i, len(parts), "section"))
    return chunks


def chunk_hybrid(note, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Strategy 3: Hybrid — section-aware with fixed-size fallback."""
    # Same as section-based but labels as hybrid
    chunks = chunk_by_section(note, chunk_size, overlap)
    for c in chunks:
        c.strategy = "hybrid"
    return chunks


# Compare strategies on one note
sample_note = notes[0]
fixed_chunks = chunk_fixed_size(sample_note)
section_chunks = chunk_by_section(sample_note)
hybrid_chunks = chunk_hybrid(sample_note)

print(f"Note: {sample_note.title} ({sample_note.char_count} chars)")
print(f"\n  Fixed-size chunks:  {len(fixed_chunks)}")
print(f"  Section-based chunks: {len(section_chunks)}")
print(f"  Hybrid chunks:      {len(hybrid_chunks)}")

print(f"\n--- Fixed-size chunk lengths ---")
for c in fixed_chunks[:5]:
    print(f"  {c.chunk_id}: {len(c.text)} chars")

print(f"\n--- Section-based chunks ---")
for c in section_chunks[:5]:
    print(f"  {c.chunk_id}: {len(c.text)} chars | Section: {c.section_heading}")


Note: Book Notes: Designing Machine Learning Systems (Chip Huyen) (1682 chars)

  Fixed-size chunks:  6
  Section-based chunks: 7
  Hybrid chunks:      7

--- Fixed-size chunk lengths ---
  b3dddf-0f36dbed: 400 chars
  b3dddf-e57bd2f9: 399 chars
  b3dddf-5d5e85e6: 400 chars
  b3dddf-8a7bd2ee: 400 chars
  b3dddf-5781fcd3: 400 chars

--- Section-based chunks ---
  b3dddf-2c0a6bac: 61 chars | Section: Book Notes: Designing Machine Learning Systems (Chip Huyen)
  b3dddf-d34757d8: 400 chars | Section: Chapter 3: Data Engineering Fundamentals
  b3dddf-c5a914ce: 170 chars | Section: Chapter 3: Data Engineering Fundamentals
  b3dddf-399b1173: 400 chars | Section: Chapter 5: Feature Engineering
  b3dddf-5e7afbcb: 201 chars | Section: Chapter 5: Feature Engineering


In [8]:
# ── Chunk all notes using hybrid strategy ──

all_chunks = []
for note in notes:
    note_chunks = chunk_by_section(note)
    all_chunks.extend(note_chunks)
    print(f"  [{note.note_id}] {note.title}: {len(note_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg chunk size: {np.mean([len(c.text) for c in all_chunks]):.0f} chars")
print(f"Size range: {min(len(c.text) for c in all_chunks)} - "
      f"{max(len(c.text) for c in all_chunks)} chars")

# Chunk distribution by note
print(f"\nChunks per note:")
chunks_per_note = Counter(c.filename for c in all_chunks)
for fn, count in chunks_per_note.most_common():
    print(f"  {fn}: {count} chunks")

  [b3dddfe60e] Book Notes: Designing Machine Learning Systems (Chip Huyen): 7 chunks
  [8527242c32] Debugging War Stories: 6 chunks
  [d61e0bfffc] Meeting Notes — March 2024: 6 chunks
  [abc74f39c4] ML Model Deployment Guide: 6 chunks
  [8efe1b8c0b] Machine Learning Fundamentals: 8 chunks
  [e062a1c4b0] Personal Goals 2024: 5 chunks
  [5a99c16a2e] Q3 2024 Project Retrospective — ML Pipeline Overhaul: 5 chunks
  [cc4a499da9] Python Async Programming Notes: 5 chunks
  [a34fa50808] Python Tips and Best Practices: 7 chunks
  [a7fa3a5f69] Research Notes: Transformers Architecture: 5 chunks

Total chunks: 60
Avg chunk size: 238 chars
Size range: 2 - 400 chars

Chunks per note:
  ml_fundamentals.md: 8 chunks
  book_notes_designing_ml_systems.md: 7 chunks
  python_tips.md: 7 chunks
  debugging_war_stories.md: 6 chunks
  meeting_notes_2024_03.md: 6 chunks
  ml_deployment_guide.md: 6 chunks
  personal_goals_2024.md: 5 chunks
  project_retro_q3.md: 5 chunks
  python_async_notes.md: 5 chunks
  res

## Step 3: Embedding

Embedding converts text into dense numerical vectors that capture semantic meaning.

**How it works:**
1. Text is tokenized into subword pieces
2. Tokens are passed through a transformer network
3. The final hidden states are pooled (mean pooling) into a single vector
4. This vector is normalized to unit length

**Why it matters for retrieval:**
- "gradient descent optimization" and "minimizing loss function" have similar embeddings
  even though they share no words
- Cosine similarity between embeddings measures semantic relatedness
- Higher similarity = more likely to answer the same question

In [9]:
# ── Build embedding index ──

class KnowledgeIndex:
    """Vector index for knowledge base chunks."""

    def __init__(self, chunks):
        self.chunks = chunks
        self.texts = [c.text for c in chunks]
        self.chunk_map = {c.chunk_id: c for c in chunks}

        if USE_SENTENCE_TRANSFORMERS and USE_CHROMA:
            self._build_chroma_index()
        else:
            self._build_tfidf_index()

    def _build_chroma_index(self):
        """ChromaDB + sentence-transformer embeddings."""
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("pkb")
        except Exception:
            pass

        self.collection = self.client.create_collection(
            name="pkb",
            metadata={"hnsw:space": "cosine"},
        )

        embeddings = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [c.chunk_id for c in self.chunks]
        metadatas = []
        for c in self.chunks:
            m = c.metadata.copy()
            # ChromaDB requires string/int/float values
            m["chunk_index"] = int(m["chunk_index"])
            m["total_chunks"] = int(m["total_chunks"])
            metadatas.append(m)

        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=self.texts,
            metadatas=metadatas,
        )
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} chunks with ChromaDB")

    def _build_tfidf_index(self):
        """TF-IDF fallback index."""
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words="english",
            ngram_range=(1, 2),
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} chunks with TF-IDF ({self.tfidf_matrix.shape[1]} features)")

    def search(self, query, top_k=TOP_K, filter_filename=None):
        """Search for relevant chunks.

        Args:
            query: Natural language query
            top_k: Number of results
            filter_filename: Optional — only search within a specific file

        Returns:
            List of (chunk, score) tuples
        """
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, filter_filename)
        else:
            return self._search_tfidf(query, top_k, filter_filename)

    def _search_chroma(self, query, top_k, filter_filename):
        where_clause = {"filename": filter_filename} if filter_filename else None
        query_emb = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(top_k, self.collection.count()),
            where=where_clause,
        )
        output = []
        for i, cid in enumerate(results["ids"][0]):
            chunk = self.chunk_map[cid]
            distance = results["distances"][0][i]
            similarity = 1.0 - distance
            output.append((chunk, similarity))
        return output

    def _search_tfidf(self, query, top_k, filter_filename):
        if filter_filename:
            candidates = [i for i, c in enumerate(self.chunks) if c.filename == filter_filename]
        else:
            candidates = list(range(len(self.chunks)))

        if not candidates:
            return []

        query_vec = self.vectorizer.transform([query])
        candidate_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, candidate_matrix).flatten()

        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[candidates[i]], float(sims[i])) for i in top_idx]


# Build the index
index = KnowledgeIndex(all_chunks)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 60 chunks with ChromaDB


## Baseline: Keyword Search

Before testing our vector-based approach, we establish a keyword search baseline.
This helps us measure the value that semantic embeddings add over simple text matching.

In [10]:
# ── Baseline: keyword search ──

def keyword_search(query, chunks, top_k=TOP_K):
    """Count query term occurrences in each chunk."""
    query_terms = set(query.lower().split())
    scored = []
    for chunk in chunks:
        text_lower = chunk.text.lower()
        hits = sum(1 for t in query_terms if t in text_lower)
        score = hits / max(len(query_terms), 1)
        scored.append((chunk, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


# Test baseline
test_q = "What is gradient descent and how does the learning rate work?"
baseline = keyword_search(test_q, all_chunks)

print(f"Query: {test_q}")
print(f"\n--- Baseline (keyword) ---")
for chunk, score in baseline[:3]:
    print(f"  Score: {score:.2f} | {chunk.citation}")
    print(f"  Preview: {chunk.text[:100]}...")
    print()

Query: What is gradient descent and how does the learning rate work?

--- Baseline (keyword) ---
  Score: 0.55 | [ml_fundamentals.md], Gradient Descent (chunk 1/3)
  Preview: Gradient descent is an optimization algorithm used to minimize a loss function.
It works by iterativ...

  Score: 0.36 | [ml_deployment_guide.md], Monitoring in Production (chunk 1/1)
  Preview: Key metrics to track:
- Prediction latency (p50, p95, p99)
- Prediction volume and error rates
- Fea...

  Score: 0.36 | [ml_fundamentals.md], Gradient Descent (chunk 2/3)
  Preview: uses one sample per step (fast but noisy)
- Mini-batch GD: uses a subset (best tradeoff, typical bat...



## Step 4: Retrieval

Now let's test semantic retrieval across different topics in our knowledge base.

In [11]:
# ── Semantic retrieval ──

test_q = "What is gradient descent and how does the learning rate work?"
results = index.search(test_q, top_k=5)

print(f"Query: {test_q}")
print(f"Backend: {index.backend}")
print(f"\n--- Semantic Results ---")
for chunk, score in results:
    print(f"  Score: {score:.3f} | {chunk.citation}")
    print(f"  Preview: {chunk.text[:120]}...")
    print()

Query: What is gradient descent and how does the learning rate work?
Backend: chroma

--- Semantic Results ---
  Score: 0.775 | [ml_fundamentals.md], Gradient Descent (chunk 1/3)
  Preview: Gradient descent is an optimization algorithm used to minimize a loss function.
It works by iteratively moving in the di...

  Score: 0.437 | [ml_fundamentals.md], Gradient Descent (chunk 2/3)
  Preview: uses one sample per step (fast but noisy)
- Mini-batch GD: uses a subset (best tradeoff, typical batch size 32-256)

Lea...

  Score: 0.300 | [ml_fundamentals.md], Machine Learning Fundamentals (chunk 1/1)
  Preview: # Machine Learning Fundamentals...

  Score: 0.296 | [book_notes_designing_ml_systems.md], Book Notes: Designing Machine Learning Systems (Chip Huyen) (chunk 1/1)
  Preview: # Book Notes: Designing Machine Learning Systems (Chip Huyen)...

  Score: 0.278 | [research_notes_transformers.md], Self-Attention Mechanism (chunk 1/1)
  Preview: Self-attention allows each position in a sequence 

In [12]:
# ── Test across different topics ──

test_queries = [
    "How do I handle missing values in features?",
    "What went wrong in the Q3 project?",
    "How does model deployment work?",
    "What are my goals for 2024?",
    "How does async programming work in Python?",
    "What is self-attention in transformers?",
    "Tell me about the timezone bug",
    "Best practices for virtual environments",
]

print("=== Multi-Topic Retrieval Test ===\n")
for q in test_queries:
    results = index.search(q, top_k=1)
    if results:
        chunk, score = results[0]
        print(f"Q: {q}")
        print(f"  -> [{score:.3f}] {chunk.filename} | {chunk.section_heading}")
        print(f"     {chunk.text[:80]}...\n")
    else:
        print(f"Q: {q} -> No results\n")

=== Multi-Topic Retrieval Test ===

Q: How do I handle missing values in features?
  -> [0.495] book_notes_designing_ml_systems.md | Chapter 5: Feature Engineering
     Good features > good algorithms. Feature engineering guidelines:
- Start with do...

Q: What went wrong in the Q3 project?
  -> [0.423] project_retro_q3.md | Q3 2024 Project Retrospective — ML Pipeline Overhaul
     # Q3 2024 Project Retrospective — ML Pipeline Overhaul...

Q: How does model deployment work?
  -> [0.625] ml_deployment_guide.md | ML Model Deployment Guide
     # ML Model Deployment Guide...

Q: What are my goals for 2024?
  -> [0.739] personal_goals_2024.md | Personal Goals 2024
     # Personal Goals 2024...

Q: How does async programming work in Python?
  -> [0.770] python_async_notes.md | Python Async Programming Notes
     # Python Async Programming Notes...

Q: What is self-attention in transformers?
  -> [0.491] research_notes_transformers.md | Research Notes: Transformers Architecture
     # Resear

## Step 5: Answer Synthesis

We build a rule-based answer synthesizer that:
1. Retrieves top-K chunks
2. Extracts the most query-relevant sentences
3. Assembles them with citations
4. Computes a confidence score

In production, you'd replace the extractive synthesizer with an LLM (GPT-4, Claude,
Llama 3) that generates natural answers grounded in the retrieved context.

In [13]:
# ── Answer synthesis ──

class PKBChat:
    """Personal Knowledge Base Chat — full QA pipeline."""

    def __init__(self, index):
        self.index = index
        self.history = []

    def ask(self, question, top_k=TOP_K, filter_filename=None, verbose=True):
        """Ask a question and get a grounded answer.

        Args:
            question: Natural language question
            top_k: Number of chunks to retrieve
            filter_filename: Optional — restrict to a specific note
            verbose: Whether to print detailed output

        Returns:
            Dict with answer, sources, confidence
        """
        # Step 1: Retrieve
        results = self.index.search(question, top_k=top_k, filter_filename=filter_filename)

        if not results:
            response = {
                "answer": "I could not find relevant information in your knowledge base.",
                "sources": [],
                "confidence": 0.0,
                "confidence_level": "NONE",
            }
            if verbose:
                print(f"Q: {question}")
                print(f"A: {response['answer']}")
            return response

        # Step 2: Extract relevant sentences
        answer_text = self._synthesize(question, results)

        # Step 3: Build citations
        sources = []
        for chunk, score in results[:3]:
            sources.append({
                "citation": chunk.citation,
                "filename": chunk.filename,
                "section": chunk.section_heading,
                "relevance": round(score, 3),
                "text_preview": chunk.text[:150],
            })

        # Step 4: Confidence
        scores = [s for _, s in results]
        confidence = self._compute_confidence(scores)

        response = {
            "answer": answer_text,
            "sources": sources,
            "confidence": confidence["score"],
            "confidence_level": confidence["level"],
        }

        # Step 5: Display
        if verbose:
            self._display(question, response, results, filter_filename)

        # Track history
        self.history.append({"question": question, "response": response})

        return response

    def _synthesize(self, question, results):
        """Extract and assemble answer from retrieved chunks."""
        q_terms = set(question.lower().split())
        answer_parts = []

        for chunk, score in results[:3]:
            sentences = re.split(r'(?<=[.!?])\s+', chunk.text)
            scored_sents = []
            for sent in sentences:
                overlap = sum(1 for t in q_terms if t in sent.lower())
                if overlap > 0 and len(sent) > 20:
                    scored_sents.append((sent, overlap))

            scored_sents.sort(key=lambda x: x[1], reverse=True)
            best = [s for s, _ in scored_sents[:2]]

            if best:
                source_tag = f"[{chunk.filename}]"
                answer_parts.append(f"{' '.join(best)} {source_tag}")

        if answer_parts:
            return "\n\n".join(answer_parts)
        else:
            top_chunk = results[0][0]
            return f"{top_chunk.text[:300]}... [{top_chunk.filename}]"

    def _compute_confidence(self, scores):
        """Compute retrieval confidence from similarity scores."""
        top = scores[0]
        avg = np.mean(scores)
        gap = scores[0] - scores[1] if len(scores) > 1 else 0.0

        # Weighted score
        conf = 0.5 * min(top / 0.7, 1.0) + 0.3 * min(avg / 0.5, 1.0) + 0.2 * min(gap / 0.15, 1.0)
        conf = round(min(conf, 1.0), 3)

        if conf >= 0.7:
            level = "HIGH"
        elif conf >= 0.4:
            level = "MEDIUM"
        else:
            level = "LOW"

        return {"score": conf, "level": level}

    def _display(self, question, response, results, filter_filename):
        """Pretty-print the response."""
        print("=" * 70)
        print(f"Q: {question}")
        if filter_filename:
            print(f"   (filtered to: {filter_filename})")
        print("=" * 70)

        print(f"\nAnswer:")
        for line in response["answer"].split("\n"):
            if line.strip():
                wrapped = textwrap.fill(line, width=68, initial_indent="  ", subsequent_indent="  ")
                print(wrapped)

        conf = response["confidence"]
        level = response["confidence_level"]
        bar = "#" * int(conf * 30)
        print(f"\nConfidence: {conf:.3f} ({level}) |{bar}|")

        print(f"\nSources:")
        for i, src in enumerate(response["sources"], 1):
            print(f"  [{i}] {src['citation']} (relevance: {src['relevance']:.3f})")

        print("=" * 70)


# Create the chat interface
chat = PKBChat(index)
print("PKB Chat ready. Knowledge base: {} notes, {} chunks".format(len(notes), len(all_chunks)))

PKB Chat ready. Knowledge base: 10 notes, 60 chunks


## Query Examples

Let's test the complete pipeline with questions spanning different topics.

In [14]:
# ── Query 1: ML Concepts ──
chat.ask("What is gradient descent and what are its variants?")

Q: What is gradient descent and what are its variants?

Answer:
  Gradient descent is an optimization algorithm used to minimize a
  loss function. It works by iteratively moving in the direction of
  steepest descent, which is
  the negative of the gradient. [ml_fundamentals.md]
  uses one sample per step (fast but noisy)
  - Mini-batch GD: uses a subset (best tradeoff, typical batch size
  32-256)
  Learning rate schedules help convergence:
  - Step decay: reduce LR by factor every N epochs
  - Cosine annealing: smooth LR reduction following cosine curve
  - Warmup: start with small LR, ramp up, then decay
  [ml_fundamentals.md]

Confidence: 0.936 (HIGH) |############################|

Sources:
  [1] [ml_fundamentals.md], Gradient Descent (chunk 1/3) (relevance: 0.761)
  [2] [ml_fundamentals.md], Gradient Descent (chunk 2/3) (relevance: 0.419)
  [3] [personal_goals_2024.md], Technical Skills (chunk 1/1) (relevance: 0.271)


{'answer': 'Gradient descent is an optimization algorithm used to minimize a loss function. It works by iteratively moving in the direction of steepest descent, which is\nthe negative of the gradient. [ml_fundamentals.md]\n\nuses one sample per step (fast but noisy)\n- Mini-batch GD: uses a subset (best tradeoff, typical batch size 32-256)\n\nLearning rate schedules help convergence:\n- Step decay: reduce LR by factor every N epochs\n- Cosine annealing: smooth LR reduction following cosine curve\n- Warmup: start with small LR, ramp up, then decay [ml_fundamentals.md]',
 'sources': [{'citation': '[ml_fundamentals.md], Gradient Descent (chunk 1/3)',
   'filename': 'ml_fundamentals.md',
   'section': 'Gradient Descent',
   'relevance': 0.761,
   'text_preview': 'Gradient descent is an optimization algorithm used to minimize a loss function.\nIt works by iteratively moving in the direction of steepest descent, w'},
  {'citation': '[ml_fundamentals.md], Gradient Descent (chunk 2/3)',
   'fi

In [15]:
# ── Query 2: Project History ──
chat.ask("What went wrong in the Q3 project retrospective?")

Q: What went wrong in the Q3 project retrospective?

Answer:
  # Q3 2024 Project Retrospective — ML Pipeline Overhaul
  [project_retro_q3.md]
  Attendees: Alice, Bob, Charlie, Diana
  Topics discussed:
  - Data warehouse migration to Snowflake is 60% complete
  - New data sources from mobile app need ingestion pipeline
  - Alice will lead the data quality dashboard initiative
  - Bob raised concern about PII in analytics tables — need masking
  Action items:
  - Alice: data quality dashboard prototype by March 15
  - Bob: PII audit of top 20 analytics t [meeting_notes_2024_03.md]
  a quality dashboard prototype by March 15
  - Bob: PII audit of top 20 analytics tables by March 10
  - Charlie: set up Snowflake staging environment by March 8
  - Diana: document current ETL jobs for migration planning
  [meeting_notes_2024_03.md]

Confidence: 0.758 (HIGH) |######################|

Sources:
  [1] [project_retro_q3.md], Q3 2024 Project Retrospective — ML Pipeline Overhaul (chunk 1/1) (relev

{'answer': '# Q3 2024 Project Retrospective — ML Pipeline Overhaul [project_retro_q3.md]\n\nAttendees: Alice, Bob, Charlie, Diana\nTopics discussed:\n- Data warehouse migration to Snowflake is 60% complete\n- New data sources from mobile app need ingestion pipeline\n- Alice will lead the data quality dashboard initiative\n- Bob raised concern about PII in analytics tables — need masking\n\nAction items:\n- Alice: data quality dashboard prototype by March 15\n- Bob: PII audit of top 20 analytics t [meeting_notes_2024_03.md]\n\na quality dashboard prototype by March 15\n- Bob: PII audit of top 20 analytics tables by March 10\n- Charlie: set up Snowflake staging environment by March 8\n- Diana: document current ETL jobs for migration planning [meeting_notes_2024_03.md]',
 'sources': [{'citation': '[project_retro_q3.md], Q3 2024 Project Retrospective — ML Pipeline Overhaul (chunk 1/1)',
   'filename': 'project_retro_q3.md',
   'section': 'Q3 2024 Project Retrospective — ML Pipeline Overhau

In [16]:
# ── Query 3: Practical Tips ──
chat.ask("How should I use virtual environments in Python?")

Q: How should I use virtual environments in Python?

Answer:
  Always use virtual environments for project isolation. Options:
  venv (built-in), conda, poetry, pdm. [python_tips.md]
  # Python Tips and Best Practices [python_tips.md]
  # Python Async Programming Notes [python_async_notes.md]

Confidence: 0.555 (MEDIUM) |################|

Sources:
  [1] [python_tips.md], Virtual Environments (chunk 1/1) (relevance: 0.465)
  [2] [python_tips.md], Python Tips and Best Practices (chunk 1/1) (relevance: 0.462)
  [3] [python_async_notes.md], Python Async Programming Notes (chunk 1/1) (relevance: 0.323)


{'answer': 'Always use virtual environments for project isolation. Options: venv (built-in), conda, poetry, pdm. [python_tips.md]\n\n# Python Tips and Best Practices [python_tips.md]\n\n# Python Async Programming Notes [python_async_notes.md]',
 'sources': [{'citation': '[python_tips.md], Virtual Environments (chunk 1/1)',
   'filename': 'python_tips.md',
   'section': 'Virtual Environments',
   'relevance': 0.465,
   'text_preview': 'Always use virtual environments for project isolation.\nOptions: venv (built-in), conda, poetry, pdm.\nPin dependencies with pip freeze or poetry.lock f'},
  {'citation': '[python_tips.md], Python Tips and Best Practices (chunk 1/1)',
   'filename': 'python_tips.md',
   'section': 'Python Tips and Best Practices',
   'relevance': 0.462,
   'text_preview': '# Python Tips and Best Practices'},
  {'citation': '[python_async_notes.md], Python Async Programming Notes (chunk 1/1)',
   'filename': 'python_async_notes.md',
   'section': 'Python Async Programming 

In [17]:
# ── Query 4: Technical Deep-Dive ──
chat.ask("How does self-attention work in transformers?")

Q: How does self-attention work in transformers?

Answer:
  Self-attention allows each position in a sequence to attend to all
  other positions. Computation: Q, K, V matrices derived from input
  embeddings. [research_notes_transformers.md]
  Transformers have no inherent notion of sequence order (unlike
  RNNs). Positional encodings add order information:
  - Sinusoidal: fixed, uses sin/cos at different frequencies
  - Learned: trainable embeddings per position (BERT, GPT)
  - RoPE (Rotary): rotates embeddings based on position (modern
  LLMs)
  - ALiBi: adds linear bias to attention scores based on distance
  [research_notes_transformers.md]

Confidence: 0.674 (MEDIUM) |####################|

Sources:
  [1] [research_notes_transformers.md], Self-Attention Mechanism (chunk 1/1) (relevance: 0.509)
  [2] [research_notes_transformers.md], Research Notes: Transformers Architecture (chunk 1/1) (relevance: 0.469)
  [3] [research_notes_transformers.md], Positional Encoding (chunk 1/1) (rele

{'answer': 'Self-attention allows each position in a sequence to attend to all other positions. Computation: Q, K, V matrices derived from input embeddings. [research_notes_transformers.md]\n\nTransformers have no inherent notion of sequence order (unlike RNNs). Positional encodings add order information:\n- Sinusoidal: fixed, uses sin/cos at different frequencies\n- Learned: trainable embeddings per position (BERT, GPT)\n- RoPE (Rotary): rotates embeddings based on position (modern LLMs)\n- ALiBi: adds linear bias to attention scores based on distance [research_notes_transformers.md]',
 'sources': [{'citation': '[research_notes_transformers.md], Self-Attention Mechanism (chunk 1/1)',
   'filename': 'research_notes_transformers.md',
   'section': 'Self-Attention Mechanism',
   'relevance': 0.509,
   'text_preview': 'Self-attention allows each position in a sequence to attend to all other positions.\nComputation: Q, K, V matrices derived from input embeddings.\nAtten'},
  {'citation': '

In [18]:
# ── Query 5: Debugging ──
chat.ask("Tell me about the timezone bug and what was the lesson learned")

Q: Tell me about the timezone bug and what was the lesson learned

Answer:
  Fix: standardize all timestamps to UTC at ingestion, convert for
  display only. Lesson: always store and process in UTC.
  [debugging_war_stories.md]
  Root cause: target leakage — a feature derived from the target
  variable was
  accidentally included in training. The feature was a status code
  that was only
  set after the event we were trying to predict.
  [debugging_war_stories.md]
  on features when domain suggests relationships
  - Use time-based features for temporal data (day of week, hour,
  lag features)
  Feature stores enable reuse across teams and prevent training-
  serving skew. [book_notes_designing_ml_systems.md]

Confidence: 0.691 (MEDIUM) |####################|

Sources:
  [1] [debugging_war_stories.md], The Timezone Bug (chunk 1/1) (relevance: 0.447)
  [2] [debugging_war_stories.md], The Leaky Validation (chunk 1/2) (relevance: 0.284)
  [3] [book_notes_designing_ml_systems.md], Chapter 5

{'answer': 'Fix: standardize all timestamps to UTC at ingestion, convert for display only. Lesson: always store and process in UTC. [debugging_war_stories.md]\n\nRoot cause: target leakage — a feature derived from the target variable was\naccidentally included in training. The feature was a status code that was only\nset after the event we were trying to predict. [debugging_war_stories.md]\n\non features when domain suggests relationships\n- Use time-based features for temporal data (day of week, hour, lag features)\n\nFeature stores enable reuse across teams and prevent training-serving skew. [book_notes_designing_ml_systems.md]',
 'sources': [{'citation': '[debugging_war_stories.md], The Timezone Bug (chunk 1/1)',
   'filename': 'debugging_war_stories.md',
   'section': 'The Timezone Bug',
   'relevance': 0.447,
   'text_preview': 'Model trained on UTC timestamps but served with local timestamps.\nPredictions were consistently wrong by 5-8 hours offset.\nRoot cause: front-end sent '}

In [19]:
# ── Query 6: Filtered search (within one note) ──
chat.ask(
    "What books are on my reading list?",
    filter_filename="personal_goals_2024.md"
)

Q: What books are on my reading list?
   (filtered to: personal_goals_2024.md)

Answer:
  - "Designing Machine Learning Systems" by Chip Huyen — DONE
  - "Machine Learning Engineering" by Andriy Burkov
  - "Building Machine Learning Pipelines" by Hannes Hapke
  - "Reliable Machine Learning" by Cathy Chen et al.
  [personal_goals_2024.md]
  - Exercise 4x per week minimum
  - Daily meditation practice (10 min)
  - Read 30 minutes before bed instead of screens
  - Take one full week vacation each quarter
  [personal_goals_2024.md]
  - Complete deep learning specialization — target: June 2024
  - Learn Rust basics for systems programming — target: August 2024
  - Build 3 end-to-end ML projects with deployment — target:
  December 2024
  - Contribute to 2 open-source projects — target: ongoing
  [personal_goals_2024.md]

Confidence: 0.283 (LOW) |########|

Sources:
  [1] [personal_goals_2024.md], Reading List (chunk 1/1) (relevance: 0.219)
  [2] [personal_goals_2024.md], Health & Wellness (

{'answer': '- "Designing Machine Learning Systems" by Chip Huyen — DONE\n- "Machine Learning Engineering" by Andriy Burkov\n- "Building Machine Learning Pipelines" by Hannes Hapke\n- "Reliable Machine Learning" by Cathy Chen et al. [personal_goals_2024.md]\n\n- Exercise 4x per week minimum\n- Daily meditation practice (10 min)\n- Read 30 minutes before bed instead of screens\n- Take one full week vacation each quarter [personal_goals_2024.md]\n\n- Complete deep learning specialization — target: June 2024\n- Learn Rust basics for systems programming — target: August 2024\n- Build 3 end-to-end ML projects with deployment — target: December 2024\n- Contribute to 2 open-source projects — target: ongoing [personal_goals_2024.md]',
 'sources': [{'citation': '[personal_goals_2024.md], Reading List (chunk 1/1)',
   'filename': 'personal_goals_2024.md',
   'section': 'Reading List',
   'relevance': 0.219,
   'text_preview': '- "Designing Machine Learning Systems" by Chip Huyen — DONE\n- "Machin

## Evaluation

We evaluate retrieval quality by checking whether the correct source note appears
in the top results for each test query.

In [20]:
# ── Evaluation: baseline vs semantic ──

eval_set = [
    {"query": "What is overfitting and how to prevent it?",
     "expected_file": "ml_fundamentals.md"},
    {"query": "How to use list comprehensions in Python?",
     "expected_file": "python_tips.md"},
    {"query": "What happened in the Q3 ML pipeline project?",
     "expected_file": "project_retro_q3.md"},
    {"query": "What does Chip Huyen say about feature engineering?",
     "expected_file": "book_notes_designing_ml_systems.md"},
    {"query": "What was discussed in the March data team meeting?",
     "expected_file": "meeting_notes_2024_03.md"},
    {"query": "How to deploy ML models to production?",
     "expected_file": "ml_deployment_guide.md"},
    {"query": "What is asyncio and when should I use it?",
     "expected_file": "python_async_notes.md"},
    {"query": "Explain multi-head attention in transformers",
     "expected_file": "research_notes_transformers.md"},
    {"query": "What are my career goals?",
     "expected_file": "personal_goals_2024.md"},
    {"query": "Tell me about the memory leak debugging story",
     "expected_file": "debugging_war_stories.md"},
]

baseline_hits = 0
semantic_hits = 0

print(f"{'Query':<55} {'Baseline':^10} {'Semantic':^10}")
print("-" * 75)

for e in eval_set:
    q = e["query"]
    expected = e["expected_file"]

    # Baseline
    b_results = keyword_search(q, all_chunks, top_k=3)
    b_hit = any(c.filename == expected for c, _ in b_results)
    baseline_hits += int(b_hit)

    # Semantic
    s_results = index.search(q, top_k=3)
    s_hit = any(c.filename == expected for c, _ in s_results)
    semantic_hits += int(s_hit)

    b_str = "HIT" if b_hit else "MISS"
    s_str = "HIT" if s_hit else "MISS"
    print(f"  {q[:53]:<55} {b_str:^10} {s_str:^10}")

n = len(eval_set)
print("-" * 75)
print(f"  {'ACCURACY':<55} {baseline_hits/n:^10.1%} {semantic_hits/n:^10.1%}")
print(f"\nBaseline: {baseline_hits}/{n}, Semantic: {semantic_hits}/{n}")

Query                                                    Baseline   Semantic 
---------------------------------------------------------------------------
  What is overfitting and how to prevent it?                 HIT        HIT    


  How to use list comprehensions in Python?                  MISS       HIT    
  What happened in the Q3 ML pipeline project?               HIT        HIT    
  What does Chip Huyen say about feature engineering?        HIT        HIT    
  What was discussed in the March data team meeting?         HIT        HIT    
  How to deploy ML models to production?                     HIT        HIT    
  What is asyncio and when should I use it?                  MISS       HIT    
  Explain multi-head attention in transformers               HIT        HIT    
  What are my career goals?                                  MISS       HIT    
  Tell me about the memory leak debugging story              HIT        HIT    
---------------------------------------------------------------------------
  ACCURACY                                                  70.0%      100.0%  

Baseline: 7/10, Semantic: 10/10


## Error Analysis

Let's examine edge cases and failure modes.

In [21]:
# ── Edge cases and failure analysis ──

edge_queries = [
    "What is the meaning of life?",           # Out of scope
    "Python",                                  # Too vague
    "How to fix bugs?",                        # Broad, matches multiple notes
    "Snowflake migration status",              # Very specific
    "What did Eve propose?",                   # Requires entity understanding
]

print("=== Edge Case Analysis ===\n")
for q in edge_queries:
    results = index.search(q, top_k=3)
    scores = [s for _, s in results]
    top_score = scores[0] if scores else 0
    sources = [c.filename for c, _ in results[:3]]

    response = chat.ask(q, verbose=False)
    conf = response["confidence"]

    print(f"Q: \"{q}\"")
    print(f"  Top score: {top_score:.3f} | Confidence: {conf:.3f} ({response['confidence_level']})")
    print(f"  Sources: {sources}")
    if response["confidence_level"] == "LOW":
        print(f"  -> Low confidence: answer may be unreliable or off-topic")
    print()

=== Edge Case Analysis ===



Q: "What is the meaning of life?"
  Top score: 0.207 | Confidence: 0.345 (LOW)
  Sources: ['personal_goals_2024.md', 'ml_deployment_guide.md', 'ml_deployment_guide.md']
  -> Low confidence: answer may be unreliable or off-topic



Q: "Python"
  Top score: 0.557 | Confidence: 0.705 (HIGH)
  Sources: ['python_tips.md', 'python_async_notes.md', 'ml_fundamentals.md']



Q: "How to fix bugs?"


  Top score: 0.234 | Confidence: 0.305 (LOW)
  Sources: ['debugging_war_stories.md', 'python_tips.md', 'python_tips.md']
  -> Low confidence: answer may be unreliable or off-topic

Q: "Snowflake migration status"
  Top score: 0.441 | Confidence: 0.637 (MEDIUM)
  Sources: ['meeting_notes_2024_03.md', 'meeting_notes_2024_03.md', 'project_retro_q3.md']



Q: "What did Eve propose?"
  Top score: 0.212 | Confidence: 0.311 (LOW)
  Sources: ['meeting_notes_2024_03.md', 'personal_goals_2024.md', 'python_async_notes.md']
  -> Low confidence: answer may be unreliable or off-topic



## Interactive Chat Session Demo

Simulating a multi-turn conversation with the knowledge base.

In [22]:
# ── Multi-turn chat session ──

session_questions = [
    "What machine learning concepts have I studied?",
    "How should I handle class imbalance?",
    "What deployment pattern is best for real-time predictions?",
    "What were the key metrics from the Q3 project?",
]

print("=== PKB Chat Session ===\n")
for q in session_questions:
    chat.ask(q)
    print()

print(f"Session history: {len(chat.history)} questions asked")

=== PKB Chat Session ===



Q: What machine learning concepts have I studied?

Answer:
  # Machine Learning Fundamentals [ml_fundamentals.md]
  - "Designing Machine Learning Systems" by Chip Huyen — DONE
  - "Machine Learning Engineering" by Andriy Burkov
  - "Building Machine Learning Pipelines" by Hannes Hapke
  - "Reliable Machine Learning" by Cathy Chen et al. - "The Staff
  Engineer's Path" by Tanya Reilly [personal_goals_2024.md]
  # Book Notes: Designing Machine Learning Systems (Chip Huyen)
  [book_notes_designing_ml_systems.md]

Confidence: 0.751 (HIGH) |######################|

Sources:
  [1] [ml_fundamentals.md], Machine Learning Fundamentals (chunk 1/1) (relevance: 0.605)
  [2] [personal_goals_2024.md], Reading List (chunk 1/1) (relevance: 0.589)
  [3] [book_notes_designing_ml_systems.md], Book Notes: Designing Machine Learning Systems (Chip Huyen) (chunk 1/1) (relevance: 0.533)

Q: How should I handle class imbalance?

Answer:
  Stratified K-fold preserves class distribution in each fold —
  essentia

Q: What were the key metrics from the Q3 project?

Answer:
  Key metrics to track:
  - Prediction latency (p50, p95, p99)
  - Prediction volume and error rates
  - Feature drift using PSI or KS statistics
  - Model performance on labeled data samples
  - Infrastructure: CPU/GPU utilization, memory, disk
  Set up alerts for anomalies in any of these metrics.
  [ml_deployment_guide.md]
  # Q3 2024 Project Retrospective — ML Pipeline Overhaul
  [project_retro_q3.md]

Confidence: 0.536 (MEDIUM) |################|

Sources:
  [1] [ml_deployment_guide.md], Monitoring in Production (chunk 1/1) (relevance: 0.413)
  [2] [project_retro_q3.md], Q3 2024 Project Retrospective — ML Pipeline Overhaul (chunk 1/1) (relevance: 0.389)
  [3] [meeting_notes_2024_03.md], March 5: Data Team Sync (chunk 2/2) (relevance: 0.339)

Session history: 15 questions asked


In [23]:
# ── Export metrics ──

metrics = {
    "project": "Personal Knowledge Base Chat",
    "knowledge_base": {
        "total_notes": len(notes),
        "total_chunks": len(all_chunks),
        "total_words": sum(n.word_count for n in notes),
        "avg_chunk_size_chars": int(np.mean([len(c.text) for c in all_chunks])),
    },
    "retrieval_backend": index.backend,
    "evaluation": {
        "total_queries": len(eval_set),
        "baseline_accuracy": baseline_hits / len(eval_set),
        "semantic_accuracy": semantic_hits / len(eval_set),
    },
    "config": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "top_k": TOP_K,
        "embedding_model": EMBEDDING_MODEL if USE_SENTENCE_TRANSFORMERS else "TF-IDF",
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\12_Personal_Knowledge_Base_Chat\metrics.json
{
  "project": "Personal Knowledge Base Chat",
  "knowledge_base": {
    "total_notes": 10,
    "total_chunks": 60,
    "total_words": 2224,
    "avg_chunk_size_chars": 237
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "total_queries": 10,
    "baseline_accuracy": 0.7,
    "semantic_accuracy": 1.0
  },
  "config": {
    "chunk_size": 400,
    "chunk_overlap": 80,
    "top_k": 5,
    "embedding_model": "all-MiniLM-L6-v2"
  }
}


## Limitations

1. **Extractive answers only** — the synthesizer selects sentences from chunks rather
   than generating natural prose. An LLM would produce much better answers.

2. **No conversation memory** — each question is independent. The system doesn't
   use previous Q&A context to improve follow-up answers.

3. **No incremental indexing** — adding a new note requires rebuilding the entire index.
   A production system would support incremental updates.

4. **Simple chunking** — section-based chunking works for well-structured markdown but
   struggles with unstructured text. Semantic chunking would help.

5. **No multi-modal support** — can't handle images, PDFs, or code files with syntax.

6. **Fixed embedding model** — using one model for all content types. Domain-specific
   models could improve retrieval for technical content.

## Common Mistakes

1. **Chunking too aggressively** — Very small chunks (< 100 chars) lose context.
   A chunk that says "use L2 regularization" without explaining what L2 is or
   why is nearly useless for answering questions.

2. **Not preserving metadata** — If chunks don't carry source information, you
   can't cite where the answer came from. Always attach metadata during chunking.

3. **Ignoring chunk overlap** — Without overlap, information at chunk boundaries
   is lost. Even 10-20% overlap significantly improves boundary retrieval.

4. **Not evaluating retrieval separately from generation** — If the retrieval
   returns wrong chunks, no amount of LLM magic will produce a correct answer.
   Always evaluate retrieval accuracy independently.

5. **Embedding the query differently than documents** — The same embedding model
   and preprocessing must be used for both documents and queries. Mismatches
   (different tokenization, casing, truncation) degrade similarity scores.

6. **Not handling out-of-scope queries** — When someone asks about something
   not in the knowledge base, the system should say "I don't know" rather than
   returning the least-irrelevant chunk.

## Mini Challenge

### Exercise 1: Add Your Own Notes
Replace the sample notes with your own `.md` or `.txt` files. Point `NOTES_DIR`
to your notes folder and re-run the notebook.

### Exercise 2: Compare Chunking Strategies
Modify the chunking to use `chunk_fixed_size` instead of `chunk_by_section`.
Re-run the evaluation. Which strategy gives better retrieval accuracy?

### Exercise 3: Tune Chunk Size
Try `CHUNK_SIZE=200` and `CHUNK_SIZE=800`. How does retrieval quality change?
Plot accuracy vs chunk size.

### Exercise 4: Add Conversation Context
Modify `PKBChat.ask()` to include the previous question in the current query
(e.g., concatenate them). Does this improve follow-up question accuracy?

### Exercise 5: Implement Reranking
After initial retrieval, use a cross-encoder to rerank:
```python
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs = [(query, chunk.text) for chunk, _ in results]
rerank_scores = reranker.predict(pairs)
```

## Production Considerations

1. **LLM-powered generation** — Use GPT-4, Claude, or a local Llama model to
   generate natural answers from retrieved context instead of extractive assembly.

2. **Incremental indexing** — Watch the notes directory for changes and update
   the index without full rebuild (ChromaDB supports upsert).

3. **Multi-format ingestion** — Support `.pdf`, `.docx`, `.html`, `.epub`,
   Notion exports, Obsidian vaults, and other knowledge management formats.

4. **Semantic chunking** — Use an embedding model to detect topic boundaries
   within documents. Split when the embedding similarity between consecutive
   sentences drops below a threshold.

5. **Hybrid search** — Combine dense embeddings with BM25 keyword search.
   Use reciprocal rank fusion to merge results from both systems.

6. **User feedback** — Let users rate answers. Use feedback to fine-tune
   the retrieval model or adjust ranking weights.

7. **Privacy** — All data stays local. For cloud deployment, encrypt at rest
   and in transit. Consider differential privacy for sensitive notes.

8. **UI** — Build a chat interface with Streamlit, Gradio, or a CLI tool
   for daily use.

## How to Improve This Project

- **HyDE (Hypothetical Document Embeddings)** — Generate a hypothetical answer
  first, then use its embedding for retrieval. Often improves recall.
- **Parent-child retrieval** — Retrieve at the sentence level but expand to
  paragraph level for context in the answer.
- **Knowledge graph** — Extract entities and relationships from notes to enable
  structured queries ("What meetings mentioned Alice?").
- **Auto-tagging** — Use an LLM to automatically tag notes with topics,
  enabling metadata-filtered search.
- **Scheduled re-indexing** — Cron job to re-index notes nightly.
- **Export as Obsidian plugin** — Package the RAG pipeline as a plugin for
  Obsidian or other note-taking apps.

## Cleanup

Remove the sample notes directory created during this notebook.

In [24]:
# ── Cleanup sample notes ──
import shutil

if os.path.exists(NOTES_DIR):
    shutil.rmtree(NOTES_DIR)
    print(f"Cleaned up {NOTES_DIR}")
else:
    print("Nothing to clean up")

Cleaned up E:\Github\Machine-Learning-Projects\GenAI\12_Personal_Knowledge_Base_Chat\_sample_notes


## Key Takeaways

1. **Ingestion is foundational** — clean, metadata-rich loading determines
   everything downstream. Garbage in = garbage out.

2. **Chunking strategy matters more than people think** — section-aware chunking
   dramatically outperforms naive fixed-size splitting for structured documents.

3. **Embeddings capture meaning, not words** — this is why "gradient descent"
   matches "optimization algorithm" but keyword search doesn't.

4. **Confidence scoring prevents bad answers** — when retrieval scores are low,
   the system should say "I'm not sure" rather than hallucinating.

5. **Source citation builds trust** — every answer should point back to the
   exact note and section it came from so users can verify.

6. **Start simple, evaluate, then improve** — TF-IDF baseline → sentence-transformers
   → reranking → LLM generation. Each step should measurably improve quality.

7. **Personal knowledge bases are a killer app for RAG** — unlike general search,
   you know exactly what's in the corpus and can evaluate precisely.